# Filter test leakage against train and validation

This notebook reproduces `remove_leaked_porseman_test_questions.py`: only leaked **test** questions are removed. To find leakage, it compares test with the combined `train + validation` reference set. Train and validation themselves are unchanged.

The outputs are `test_clean` and a combined corpus of `train + validation + test_clean`. Recall and MRR are evaluated only with test_clean queries.

In [ ]:
# Paths, model, and encoding configuration
from pathlib import Path
import numpy as np
import pandas as pd
from FlagEmbedding import BGEM3FlagModel

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'benchmark_test_leakage_filtered'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'porseman_train.csv'
VALIDATION_PATH = DATA_DIR / 'porseman_validation.csv'
TEST_PATH = DATA_DIR / 'porseman_test.csv'
FILTER_MODEL = 'BAAI/bge-m3'
THRESHOLD = 0.90
DEVICE = 'cuda:0'
# Keep model-specific batch sizes separate: model memory requirements differ.
BGE_ENCODE_BATCH_SIZE = 16
JINA_ENCODE_BATCH_SIZE = 2
SNOWFLAKE_ENCODE_BATCH_SIZE = 2
MAX_LENGTH = 1024


## Load data splits

Load train, validation, and test; combine train and validation as the reference set.

In [ ]:
# Load splits and build the reference set
def read_split(path, split):
    df = pd.read_csv(path, encoding='utf-8-sig', dtype=str).fillna('')
    required = {'id', 'question', 'content_text'}
    if required - set(df.columns) or not df.id.is_unique:
        raise ValueError(f'Invalid split: {path}')
    return df.assign(source_split=split)

train = read_split(TRAIN_PATH, 'train')
validation = read_split(VALIDATION_PATH, 'validation')
test = read_split(TEST_PATH, 'test')
reference = pd.concat([train, validation], ignore_index=True)
assert not set(reference.id) & set(test.id)
pd.DataFrame({'split': ['train', 'validation', 'test', 'train + validation'],
              'rows': [len(train), len(validation), len(test), len(reference)]})


## Encode questions and compute semantic similarity

Encode test and reference questions, then compute their cosine similarity.

In [ ]:
# Encode questions and calculate cosine similarity
model_args = {'use_fp16': DEVICE != 'cpu', 'pooling_method': 'cls'}
if DEVICE:
    model_args['devices'] = [DEVICE]
model = BGEM3FlagModel(FILTER_MODEL, **model_args)

def encode(questions):
    vecs = model.encode_queries(questions.tolist(), batch_size=BGE_ENCODE_BATCH_SIZE, max_length=MAX_LENGTH,
        return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
    vecs = np.asarray(vecs, dtype=np.float32)
    return vecs / np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-12)

reference_vectors = encode(reference.question)
test_vectors = encode(test.question)
scores = test_vectors @ reference_vectors.T
nearest_index = scores.argmax(axis=1)
max_similarity = scores[np.arange(len(test)), nearest_index]


## Build the leakage audit

Record the nearest reference question and the keep/remove decision for every test question.

In [ ]:
# Build an auditable keep-or-remove decision for each test question
audit = test.copy()
audit['nearest_reference_id'] = reference.iloc[nearest_index].id.to_numpy()
audit['nearest_reference_split'] = reference.iloc[nearest_index].source_split.to_numpy()
audit['nearest_reference_question'] = reference.iloc[nearest_index].question.to_numpy()
audit['cosine_similarity'] = max_similarity
audit['meets_threshold'] = audit.cosine_similarity >= THRESHOLD
audit['remove_from_test'] = audit.meets_threshold
audit['reason'] = np.where(audit.meets_threshold, f'cosine>={THRESHOLD:.2f}', 'keep')
audit.sort_values('cosine_similarity', ascending=False)[['id', 'question', 'nearest_reference_split', 'nearest_reference_question', 'cosine_similarity', 'reason']].head(30)


## Review threshold impact

Compare how many test questions would be removed at different similarity thresholds.

In [ ]:
# Compare removal counts across candidate thresholds
thresholds = [0.80, 0.85, 0.88, 0.90, 0.92, 0.95]
pd.DataFrame({'threshold': thresholds,
              'test_rows_removed': [(audit.cosine_similarity >= t).sum() for t in thresholds],
              'removed_fraction': [(audit.cosine_similarity >= t).mean() for t in thresholds]})


## Save outputs

This cell never overwrites input files. `best_reference_match...` is the report equivalent used by the existing script; `porseman_test_leakage_filtered.csv` is the test split after removal.

In [ ]:
# Save the audit, filtered test split, and combined evaluation corpus
audit.to_csv(OUTPUT_DIR / 'best_reference_match_for_each_test_question.csv', index=False, encoding='utf-8-sig')
audit.loc[audit.remove_from_test].to_csv(OUTPUT_DIR / 'removed_test_questions.csv', index=False, encoding='utf-8-sig')
test_clean = audit.loc[~audit.remove_from_test, ['id', 'question', 'content_text', 'source_split']].copy()
test_clean.to_csv(OUTPUT_DIR / 'porseman_test_leakage_filtered.csv', index=False, encoding='utf-8-sig')
benchmark_corpus = pd.concat([train, validation, test_clean], ignore_index=True)
benchmark_corpus.to_csv(OUTPUT_DIR / 'benchmark_corpus_all_splits.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'item': ['original test', 'removed test', 'clean test', 'whole corpus'],
              'rows': [len(test), audit.remove_from_test.sum(), len(test_clean), len(benchmark_corpus)]})


## Prepare fixed evaluation inputs

Load the filtered test split and the full corpus once. Every evaluated model will use these exact inputs.

In [ ]:
# Load and validate the fixed queries, corpus, and relevance mapping
EVAL_TEST_PATH = OUTPUT_DIR / 'porseman_test_leakage_filtered.csv'
EVAL_CORPUS_PATH = OUTPUT_DIR / 'benchmark_corpus_all_splits.csv'
eval_test = pd.read_csv(EVAL_TEST_PATH, encoding='utf-8-sig', dtype=str).fillna('')
eval_corpus = pd.read_csv(EVAL_CORPUS_PATH, encoding='utf-8-sig', dtype=str).fillna('')

required_columns = {'id', 'question', 'content_text'}
if required_columns - set(eval_test.columns) or required_columns - set(eval_corpus.columns):
    raise ValueError('Evaluation inputs must contain id, question, and content_text.')
if not eval_corpus.id.is_unique or not set(eval_test.id).issubset(set(eval_corpus.id)):
    raise ValueError('Corpus IDs must be unique and must contain every test ID.')

test_queries = eval_test.question.tolist()
corpus_documents = eval_corpus.content_text.tolist()
corpus_index_by_id = pd.Series(np.arange(len(eval_corpus)), index=eval_corpus.id)
relevant_indices = corpus_index_by_id.loc[eval_test.id].to_numpy()
pd.DataFrame({'item': ['test queries', 'corpus documents'], 'count': [len(test_queries), len(corpus_documents)]})


## Define common retrieval metrics

Use one exact-search implementation of Recall@1, Recall@5, and MRR@10 for every model.

In [ ]:
# Calculate exact retrieval metrics from normalized query and corpus embeddings
def l2_normalize(embeddings):
    embeddings = np.asarray(embeddings, dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError('The model produced a zero-length embedding.')
    return embeddings / norms

def calculate_retrieval_metrics(query_embeddings, corpus_embeddings, relevant_indices, search_batch_size=64):
    query_embeddings = l2_normalize(query_embeddings)
    corpus_embeddings = l2_normalize(corpus_embeddings)
    if len(query_embeddings) != len(relevant_indices):
        raise ValueError('Each query must have one relevant corpus index.')
    top_k = min(10, len(corpus_embeddings))
    recall_at_1 = recall_at_5 = 0
    reciprocal_rank_at_10 = 0.0
    corpus_t = np.ascontiguousarray(corpus_embeddings.T)

    for start in range(0, len(query_embeddings), search_batch_size):
        end = min(start + search_batch_size, len(query_embeddings))
        scores = query_embeddings[start:end] @ corpus_t
        ranked = np.argpartition(-scores, kth=top_k - 1, axis=1)[:, :top_k]
        ranked_scores = np.take_along_axis(scores, ranked, axis=1)
        ranked = np.take_along_axis(ranked, np.argsort(-ranked_scores, axis=1), axis=1)
        for offset, candidates in enumerate(ranked):
            hit = np.flatnonzero(candidates == relevant_indices[start + offset])
            if len(hit):
                rank = int(hit[0]) + 1
                recall_at_1 += rank == 1
                recall_at_5 += rank <= 5
                reciprocal_rank_at_10 += 1 / rank

    query_count = len(query_embeddings)
    return {'recall@1': recall_at_1 / query_count, 'recall@5': recall_at_5 / query_count,
            'mrr@10': reciprocal_rank_at_10 / query_count}


## Configure models for comparison

List every model once with its encoder family and model-specific batch size.

In [ ]:
# Define the models and encoding settings used in the benchmark
BASE_BGE_MODEL = 'BAAI/bge-m3'
FINE_TUNED_BGE_MODEL = '/home/rahnema/aahmadi/finetune-embedding-models/outputs/bge-m3-porseman-minimal'
JINA_MODEL = 'jinaai/jina-embeddings-v3'
SNOWFLAKE_MODEL = 'Snowflake/snowflake-arctic-embed-l-v2.0'

MODEL_SPECS = [
    {'label': 'base_bge_m3', 'model': BASE_BGE_MODEL, 'family': 'bge', 'batch_size': BGE_ENCODE_BATCH_SIZE},
    {'label': 'fine_tuned_bge_m3', 'model': FINE_TUNED_BGE_MODEL, 'family': 'bge', 'batch_size': BGE_ENCODE_BATCH_SIZE},
    {'label': 'jina_embeddings_v3', 'model': JINA_MODEL, 'family': 'jina', 'batch_size': JINA_ENCODE_BATCH_SIZE},
    {'label': 'snowflake_arctic_embed_l_v2', 'model': SNOWFLAKE_MODEL, 'family': 'snowflake', 'batch_size': SNOWFLAKE_ENCODE_BATCH_SIZE},
]
pd.DataFrame(MODEL_SPECS)


## Define model-specific encoders

Use each model with its recommended query and document encoding mode.

In [ ]:
# Encode queries and documents with the correct interface for each model family
from sentence_transformers import SentenceTransformer

def encode_bge(model_name_or_path, batch_size):
    encoder_args = {'use_fp16': DEVICE != 'cpu', 'pooling_method': 'cls'}
    if DEVICE:
        encoder_args['devices'] = [DEVICE]
    encoder = BGEM3FlagModel(str(model_name_or_path), **encoder_args)
    queries = encoder.encode_queries(test_queries, batch_size=batch_size, max_length=MAX_LENGTH,
        return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
    documents = encoder.encode_corpus(corpus_documents, batch_size=batch_size, max_length=MAX_LENGTH,
        return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
    return np.asarray(queries), np.asarray(documents)

def encode_sentence_transformer(model_name, batch_size, family):
    encoder = SentenceTransformer(model_name, device=DEVICE, trust_remote_code=(family == 'jina'))
    if family == 'jina':
        queries = encoder.encode(test_queries, batch_size=batch_size, convert_to_numpy=True,
            show_progress_bar=True, task='retrieval.query', prompt_name='retrieval.query')
        documents = encoder.encode(corpus_documents, batch_size=batch_size, convert_to_numpy=True,
            show_progress_bar=True, task='retrieval.passage', prompt_name='retrieval.passage')
    elif family == 'snowflake':
        queries = encoder.encode(test_queries, batch_size=batch_size, convert_to_numpy=True,
            show_progress_bar=True, prompt_name='query')
        documents = encoder.encode(corpus_documents, batch_size=batch_size, convert_to_numpy=True,
            show_progress_bar=True)
    else:
        raise ValueError(f'Unsupported SentenceTransformer family: {family}')
    return np.asarray(queries), np.asarray(documents)


## Run the benchmark

Evaluate every configured model on the same filtered test queries and full corpus.

In [ ]:
# Evaluate all models and save one comparable metrics table
import gc
import json
import torch

# The filtering model is no longer needed during evaluation.
if 'model' in globals():
    del model
gc.collect()
if DEVICE and DEVICE.startswith('cuda'):
    torch.cuda.empty_cache()

evaluation_results = []
for spec in MODEL_SPECS:
    print(f"Evaluating {spec['label']} with batch size {spec['batch_size']} ...")
    if spec['family'] == 'bge':
        query_embeddings, corpus_embeddings = encode_bge(spec['model'], spec['batch_size'])
    else:
        query_embeddings, corpus_embeddings = encode_sentence_transformer(
            spec['model'], spec['batch_size'], spec['family']
        )
    metrics = calculate_retrieval_metrics(query_embeddings, corpus_embeddings, relevant_indices)
    evaluation_results.append({**spec, **metrics, 'query_count': len(test_queries),
                               'corpus_document_count': len(corpus_documents)})
    del query_embeddings, corpus_embeddings
    gc.collect()
    if DEVICE and DEVICE.startswith('cuda'):
        torch.cuda.empty_cache()

results_df = pd.DataFrame(evaluation_results)
results_df.to_csv(OUTPUT_DIR / 'evaluation_metrics.csv', index=False, encoding='utf-8-sig')
(OUTPUT_DIR / 'evaluation_metrics.json').write_text(
    json.dumps(evaluation_results, ensure_ascii=False, indent=2), encoding='utf-8'
)
results_df[['label', 'recall@1', 'recall@5', 'mrr@10', 'query_count', 'corpus_document_count']]


## Review model comparison

Rank models by MRR@10 and measure fine-tuning gains against the BGE-M3 baseline.

In [ ]:
# Display ranked metrics and fine-tuning improvements in percentage points
metric_columns = ['recall@1', 'recall@5', 'mrr@10']
comparison = results_df[['label', *metric_columns]].sort_values('mrr@10', ascending=False).copy()
for column in metric_columns:
    comparison[column] = (comparison[column] * 100).round(2)
display(comparison.rename(columns={'recall@1': 'Recall@1 (%)', 'recall@5': 'Recall@5 (%)',
                                   'mrr@10': 'MRR@10 (%)'}))

baseline = results_df.set_index('label').loc['base_bge_m3', metric_columns]
fine_tuned = results_df.set_index('label').loc['fine_tuned_bge_m3', metric_columns]
improvement = ((fine_tuned - baseline) * 100).round(2).rename('fine_tuned_minus_baseline_pp')
display(improvement.to_frame())


## Export an HTML model-comparison report

Create a standalone HTML table with the shared evaluation setup and all model metrics.

In [ ]:
# Write a standalone HTML comparison report
from html import escape
from datetime import datetime

html_results = results_df.copy()
baseline_mrr = html_results.loc[html_results.label == 'base_bge_m3', 'mrr@10'].iloc[0]
html_results['delta_mrr_vs_base_pp'] = (html_results['mrr@10'] - baseline_mrr) * 100
html_results = html_results.sort_values('mrr@10', ascending=False)
table_rows = ''.join(
    f"<tr><td>{escape(str(row.label))}</td><td>{row['recall@1']:.2%}</td>"
    f"<td>{row['recall@5']:.2%}</td><td>{row['mrr@10']:.2%}</td>"
    f"<td>{row.delta_mrr_vs_base_pp:+.2f} pp</td></tr>"
    for _, row in html_results.iterrows()
)
generated_at = datetime.now().astimezone().isoformat(timespec='seconds')
report_html = f'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Embedding Model Comparison</title><style>
body{{font:16px/1.5 Arial,sans-serif;margin:40px;background:#f7f8fa;color:#18212f}}
main{{max-width:1000px;margin:auto;background:#fff;padding:32px;border-radius:12px;box-shadow:0 4px 20px #0001}}
h1{{margin-top:0}} .meta{{color:#536274}} table{{width:100%;border-collapse:collapse;margin-top:24px}}
th,td{{padding:12px;text-align:left;border-bottom:1px solid #dce1e8}} th{{background:#f1f4f8}}
tr:first-child td{{font-weight:700}} code{{background:#f1f4f8;padding:2px 5px;border-radius:4px}}
</style></head><body><main>
<h1>Embedding Model Comparison</h1>
<p class="meta">Generated: {escape(generated_at)}<br>Filter model: {escape(FILTER_MODEL)}; cosine threshold: {THRESHOLD:.2f}<br>
Test queries: {len(test_queries):,}; corpus documents: {len(corpus_documents):,}</p>
<p>All models use the same filtered test split and combined corpus. Delta MRR is measured against <code>base_bge_m3</code>.</p>
<table><thead><tr><th>Model</th><th>Recall@1</th><th>Recall@5</th><th>MRR@10</th><th>Delta MRR vs. base</th></tr></thead>
<tbody>{table_rows}</tbody></table></main></body></html>'''
html_report_path = OUTPUT_DIR / 'model_comparison.html'
html_report_path.write_text(report_html, encoding='utf-8')
print(html_report_path)
